**Fig. 6 and 7, step 3 (notebook 03): draw the shape panels (shape UMAPs, cluster composition, median cells and mesh galleries)**

In [ ]:
# Working directory: data/fig6_7/ under the repository root, where the Fig. 6 and 7 deposit folder is downloaded
# (all_stacks_features.csv, workspace/ = the plugin's Step 4 workspace with the 14 stacks (13 analysed) and their meshes,
# run/ = what these notebooks write and read back, figures/ = the saved panels).
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os, sys
from pathlib import Path
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig6_7")
print("working directory:", os.getcwd())

# EpiCycle Morphospace Visualizations

Styled to match the aesthetic of `figure2_main.py` and `epicycle_benchmark_main.py`.
Self-contained — no external module imports required beyond the standard stack.

**Inputs**: `all_features_multi_leiden.csv` with columns including
`umap_1, umap_2, stack_name, donor, tissue, pattern, timepoint, leiden_r{resolution}`.

**Structure**:
1. Data loading
2. Palettes and helpers (run once)
3. Plot functions (run once)
4. One cell per visualization — run independently after the above

---


## 1. Data loading

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# Update path as needed
df = pd.read_csv('run/all_features_multi_leiden.csv')

# Leiden resolution for cluster-based plots
RESOLUTION = 1.0

print(f"Loaded {len(df):,} cells across {df['stack_name'].nunique()} samples")
print(f"Clusters at r={RESOLUTION}: {df[f'leiden_r{RESOLUTION}'].nunique()}")

## 2. Palettes, metadata, and helpers

Typography ladder, color palettes, sample ordering. Run this cell once.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go


# ═══════════════════════════════════════════════════════════════════════
# TYPOGRAPHY (matches figure2_main.py / epicycle_benchmark_main.py)
# ═══════════════════════════════════════════════════════════════════════
FONT_FAMILY       = "Helvetica Neue, Arial"
FONT_AXIS_TITLE   = 13
FONT_TICK         = 11
FONT_LEGEND       = 10.5
FONT_LEGEND_TITLE = 11


# ═══════════════════════════════════════════════════════════════════════
# PALETTES — Paul Tol qualitative, consistent with your benchmark figures
# ═══════════════════════════════════════════════════════════════════════

# Canonical sample order, grouped by tissue context
STACK_ORDER = [
    # back (trunk skin -- its own region, not the hand)
    "(13 wks) Back skin x40-Smooth05_Resampled.tif",
    "(14 wks) back skin x40-Smooth05_Resampled.tif",
    "(12 wks) back skin x40-Smooth05_Resampled.tif",
    # front skin
    "(13 wks) Front skin x40-Smooth05_Resampled.tif",
    "(12 wks) front skin x40-Smooth05_Resampled.tif",
    # dorsal (back of the digit)
    "(13 wks) D3L Dorsal skin x40-Smooth05_Resampled.tif",
    "(14 wks) D3L dorsal x40-Smooth05_Resampled.tif",
    "(12 wks) D3L dorsal x40 stack-Smooth05_Resampled.tif",
    # volar, unpatterned
    "(14 wks) digit 5 unpatterned skin x40 stack-Smooth05_Resampled.tif",
    "(13 wks) D3L Ventral unpatterned skin x40-Smooth05_Resampled.tif",
    # volar, ridged
    "(13 wks) D3L ventral ridges x40-Smooth05_Resampled.tif",
    "(12 wks) D2L ventral tip x40-Smooth05_Resampled.tif",
    # volar, whorl
    "(14 wks) ventral whorl x40-Smooth05_Resampled.tif",
]

# Short names: strip the processing suffix and the objective for display
SHORT = {s: s.replace("-Smooth05_Resampled.tif", "").replace(" x40 stack", "").replace(" x40", "") for s in STACK_ORDER}

# Sample palette — hue families mirror biological grouping:
#   greens       = back skin
#   magentas     = front skin
#   teals        = dorsal digit
#   yellows      = volar unpatterned
#   oranges/reds = volar ridged
#   blue         = volar whorl
SAMPLE_PALETTE = {
    # Back skin: greens
    "(13 wks) Back skin":                    "#117733",
    "(14 wks) back skin":                    "#66BB77",
    "(12 wks) back skin":                    "#228833",
    # Dorsal digit: teals
    "(13 wks) D3L Dorsal skin":              "#44AA99",
    "(14 wks) D3L dorsal":              "#88CCAA",
    "(12 wks) D3L dorsal":              "#99DDBB",
    # Volar unpatterned: yellows
    "(14 wks) digit 5 unpatterned skin":      "#DDCC77",
    "(13 wks) D3L Ventral unpatterned skin": "#BB9944",
    # Volar ridged: oranges/reds
    "(13 wks) D3L ventral ridges":      "#CC3311",
    "(12 wks) D2L ventral tip":         "#EE7733",
    # Volar whorl: blue
    "(14 wks) ventral whorl":           "#0077BB",
    # Front skin: magentas
    "(13 wks) Front skin":                   "#AA3377",
    "(12 wks) front skin":                   "#EE3377",
}

# Pattern palette — matches the hue-family logic of SAMPLE_PALETTE
PATTERN_PALETTE = {
    "back":        "#117733",   # green   -- the back-skin samples' hue family
    "front":       "#AA3377",   # magenta -- the front-skin samples' hue family
    "dorsal":      "#44AA99",   # teal    -- the dorsal-digit samples' hue family
    "unpatterned": "#DDCC77",
    "ridged":      "#CC3311",
    "whorl":       "#0077BB",
}
PATTERN_ORDER = ["back", "front", "dorsal", "unpatterned", "ridged", "whorl"]

# Donor (embryo) palette — Paul Tol bright
DONOR_PALETTE = {
    "donor1": "#332288",
    "donor2": "#CC6677",
    "donor3": "#44AA99",
    "donor4": "#88CCEE",
    "donor5": "#DDCC77",
}

TISSUE_PALETTE = {
    "back":   "#117733",
    "front":  "#AA3377",
    "dorsal": "#44AA99",
    "volar":  "#CC3311",
}
TISSUE_ORDER = ["back", "front", "dorsal", "volar"]

# Timepoint — ordered chronologically with a sequential ramp
#   (viridis-like, perceptually uniform). Time is continuous-ish, so
#   order matters more than distinctness.
TIMEPOINT_ORDER = ["12wk", "13wk", "14wk"]
TIMEPOINT_PALETTE = {
    "12wk": "#7AD151",  # yellow-green
    "13wk": "#22A884",  # teal
    "14wk": "#414487",  # deep indigo (latest)
}

# Cluster palette — Paul Tol qualitative, repeats gracefully for 16+ clusters
_CLUSTER_BASE = [
    "#0077BB", "#EE7733", "#009988", "#CC3311", "#EE3377",
    "#33BBEE", "#AA3377", "#BB5566", "#44AA99", "#DDCC77",
    "#882255", "#117733", "#88CCEE", "#332288", "#CC6677",
    "#EE9944", "#99DDBB", "#AA4499", "#661100", "#6699CC",
    "#888888", "#D4B483",
]


def build_cluster_cmap(cluster_ids):
    """{cluster_id: hex_color}. Handles up to 22 distinctly; cycles beyond."""
    ids = sorted(set(int(c) for c in cluster_ids))
    return {c: _CLUSTER_BASE[i % len(_CLUSTER_BASE)] for i, c in enumerate(ids)}


# ═══════════════════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════════════════
def _apply_umap_axes(fig):
    """
    UMAP coordinates are unitless and arbitrary — no tick labels, no axis
    line. Subtle gridlines stay for visual anchoring without distraction.
    """
    axkw = dict(
        showgrid=True,
        gridcolor="rgba(200,200,200,0.25)",
        gridwidth=0.5,
        zeroline=False,
        showticklabels=False, ticks="",
        showline=False,
        title_font=dict(size=FONT_AXIS_TITLE, family=FONT_FAMILY),
    )
    fig.update_xaxes(title_text="UMAP 1", **axkw)
    fig.update_yaxes(title_text="UMAP 2", **axkw)


def _base_layout(width, height, legend_title=None, show_legend=True):
    return dict(
        plot_bgcolor="#fff", paper_bgcolor="#fff",
        width=width, height=height,
        margin=dict(l=65, r=180, t=30, b=60),  # r=180 so external legends don't clip in SVG
        font=dict(family=FONT_FAMILY),
        showlegend=show_legend,
        legend=dict(
            font=dict(size=FONT_LEGEND, family=FONT_FAMILY),
            title=dict(
                text=legend_title or "",
                font=dict(size=FONT_LEGEND_TITLE, family=FONT_FAMILY),
            ),
            tracegroupgap=4,
            x=1.01, y=0.98, xanchor="left",
            bgcolor="rgba(255,255,255,0.9)",
            itemsizing="constant",
        ),
    )


def _ensure_sample_short(df):
    """Add the sample_short column if it's missing (using STACK_ORDER's SHORT)."""
    if "sample_short" not in df.columns:
        df = df.copy()
        df["sample_short"] = df["stack_name"].map(SHORT)
    return df


# -----------------------------------------------------------------------
# Sample metadata -- the SAME map as notebook 01.
# 'back' and 'front' skin are regions of their own (not 'dorsal' / 'volar').
# Expected stacks per tissue: volar 5, dorsal 3, back 3, front 2.
# apply_metadata() re-stamps donor / timepoint / tissue / pattern from this
# map, so a CSV or pickle written before the change still gets the right
# labels (and the right overlay colours).
# -----------------------------------------------------------------------
META_MAP = {
    '(14 wks) digit 5 unpatterned skin x40 stack-Smooth05_Resampled.tif':
        {'donor': 'donor1', 'timepoint': '14wk', 'tissue': 'volar', 'pattern': 'unpatterned'},
    '(13 wks) D3L ventral ridges x40-Smooth05_Resampled.tif':
        {'donor': 'donor2', 'timepoint': '13wk', 'tissue': 'volar', 'pattern': 'ridged'},
    '(13 wks) D3L Ventral unpatterned skin x40-Smooth05_Resampled.tif':
        {'donor': 'donor2', 'timepoint': '13wk', 'tissue': 'volar', 'pattern': 'unpatterned'},
    '(13 wks) D3L Dorsal skin x40-Smooth05_Resampled.tif':
        {'donor': 'donor2', 'timepoint': '13wk', 'tissue': 'dorsal', 'pattern': 'dorsal'},
    '(13 wks) Back skin x40-Smooth05_Resampled.tif':
        {'donor': 'donor2', 'timepoint': '13wk', 'tissue': 'back', 'pattern': 'back'},
    '(13 wks) Front skin x40-Smooth05_Resampled.tif':
        {'donor': 'donor2', 'timepoint': '13wk', 'tissue': 'front', 'pattern': 'front'},
    '(14 wks) ventral whorl x40-Smooth05_Resampled.tif':
        {'donor': 'donor3', 'timepoint': '14wk', 'tissue': 'volar', 'pattern': 'whorl'},
    '(14 wks) back skin x40-Smooth05_Resampled.tif':
        {'donor': 'donor3', 'timepoint': '14wk', 'tissue': 'back', 'pattern': 'back'},
    '(14 wks) D3L dorsal x40-Smooth05_Resampled.tif':
        {'donor': 'donor3', 'timepoint': '14wk', 'tissue': 'dorsal', 'pattern': 'dorsal'},
    '(12 wks) D2L ventral tip x40-Smooth05_Resampled.tif':
        {'donor': 'donor4', 'timepoint': '12wk', 'tissue': 'volar', 'pattern': 'ridged'},
    '(12 wks) back skin x40-Smooth05_Resampled.tif':
        {'donor': 'donor4', 'timepoint': '12wk', 'tissue': 'back', 'pattern': 'back'},
    '(12 wks) front skin x40-Smooth05_Resampled.tif':
        {'donor': 'donor4', 'timepoint': '12wk', 'tissue': 'front', 'pattern': 'front'},
    '(12 wks) D3L dorsal x40 stack-Smooth05_Resampled.tif':
        {'donor': 'donor5', 'timepoint': '12wk', 'tissue': 'dorsal', 'pattern': 'dorsal'},
}
META_KEYS = ("donor", "timepoint", "tissue", "pattern")


def apply_metadata(df):
    """Re-stamp donor / timepoint / tissue / pattern from META_MAP.
    Prints how many rows changed per column and any stack not in the map
    (those rows keep whatever labels they had)."""
    df = df.copy()
    unmapped = sorted(set(df["stack_name"].unique()) - set(META_MAP))
    for key in META_KEYS:
        new = df["stack_name"].map(lambda s, k=key: META_MAP.get(s, {}).get(k))
        old = df[key] if key in df.columns else None
        keep = new.isna()
        if old is not None:
            new = new.where(~keep, old)
            n_changed = int((new.astype(str) != old.astype(str)).sum())
        else:
            n_changed = len(df)
        df[key] = new
        if n_changed:
            print(f"[metadata] {key}: {n_changed:,} rows relabelled from META_MAP")
    if unmapped:
        print(f"[metadata] WARNING stacks not in META_MAP (labels left as they were): {unmapped}")
    for key in ("tissue", "pattern"):
        stacks = df.groupby(key)["stack_name"].nunique()
        print(f"[metadata] {key}: " + ", ".join(
            f"{v} {stacks[v]} stacks / {n:,} cells"
            for v, n in df[key].value_counts().items()))
    return df


# -----------------------------------------------------------------------
# Grouped scatter -- how the points of different groups are layered
#   "intermix" : every point in ONE layer, in a random (seeded) order, so no
#                group sits on top of another.  Default.
#   "stacked"  : one layer per group, drawn group after group (the old look:
#                the group drawn last covers the others where they overlap).
# Every categorical UMAP function takes draw= / seed= to override per figure.
# -----------------------------------------------------------------------
SCATTER_DRAW = "intermix"
SCATTER_SEED = 0


def _ordered_present(values, order):
    """`order` filtered to the values present, then any value missing from
    `order` (sorted) -- so a group is never silently dropped from a plot."""
    present = set(values)
    extras = sorted(present - set(order), key=str)
    if extras:
        print(f"[plot] not in the order list, drawn last: {extras}")
    return [v for v in order if v in present] + extras


def _grouped_scatter(fig, d, x_col, y_col, group_col, groups, colors, *,
                     names=None, hover=None, legendgroups=None,
                     marker_size=6, opacity=0.8,
                     draw=None, seed=None, biggest_first=True):
    """Add a categorical scatter of `d` to `fig`.

    groups       : group values, in legend order (other values are skipped)
    colors       : {group: hex}; a group missing from it is grey (#888)
    names        : {group: legend text}            (default str(group))
    hover        : {group: hover text, no <extra>} (default "<b>name</b><br>n = N")
    legendgroups : {group: plotly legendgroup}     (optional)
    draw         : "intermix" | "stacked"           (default SCATTER_DRAW)
    seed         : shuffle seed for "intermix"      (default SCATTER_SEED)
    biggest_first: "stacked" only -- draw the biggest group first, so small
                   groups end up on top (as before); False = legend order.

    In "intermix" the points are one trace and the legend entries are
    empty marker-only traces, so clicking a legend entry does not hide its
    points (use draw="stacked" for that).
    """
    draw = SCATTER_DRAW if draw is None else draw
    seed = SCATTER_SEED if seed is None else seed
    if draw not in ("intermix", "stacked"):
        raise ValueError(f"draw must be 'intermix' or 'stacked', got {draw!r}")
    names, hover, legendgroups = names or {}, hover or {}, legendgroups or {}

    counts = d[group_col].value_counts()
    groups = [g for g in groups if counts.get(g, 0) > 0]
    name_of = {g: names.get(g, str(g)) for g in groups}
    hover_of = {g: hover.get(g, f"<b>{name_of[g]}</b><br>n = {int(counts[g]):,}")
                for g in groups}
    base = dict(size=marker_size, opacity=opacity, line=dict(width=0))
    tmpl = "%{x:.2f}, %{y:.2f}<extra></extra>"

    if draw == "stacked":
        order = (sorted(groups, key=lambda g: -counts[g]) if biggest_first
                 else list(groups))
        for g in order:
            sub = d[d[group_col] == g]
            fig.add_trace(go.Scatter(
                x=sub[x_col], y=sub[y_col], mode="markers",
                marker=dict(base, color=colors.get(g, "#888")),
                name=name_of[g], legendgroup=legendgroups.get(g),
                legendrank=groups.index(g),
                hovertemplate=hover_of[g] + "<br>" + tmpl,
            ))
        return fig

    # intermix: one shuffled layer + legend-only entries
    sub = d[d[group_col].isin(groups)]
    sub = sub.iloc[np.random.default_rng(seed).permutation(len(sub))]
    grp = sub[group_col]
    fig.add_trace(go.Scatter(
        x=sub[x_col].to_numpy(), y=sub[y_col].to_numpy(), mode="markers",
        marker=dict(base, color=grp.map(lambda g: colors.get(g, "#888")).to_numpy()),
        customdata=grp.map(hover_of).to_numpy(),
        hovertemplate="%{customdata}<br>" + tmpl,
        name="all points", showlegend=False,
    ))
    for rank, g in enumerate(groups):
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode="markers",
            marker=dict(base, color=colors.get(g, "#888")),
            name=name_of[g], legendgroup=legendgroups.get(g),
            legendrank=rank, hoverinfo="skip", showlegend=True,
        ))
    return fig


# Re-stamp the metadata on the df loaded above (section 1)
if "df" in globals():
    df = apply_metadata(df)


## A4-aware figure export (run once)

`save_fig(fig, name, mm=...)` writes an Illustrator-editable **SVG** (vector) and
a high-DPI **PNG** sidecar. The `mm` argument sizes the figure for an **A4
Illustrator artboard** at 100%. Requires `kaleido` (`pip install -U kaleido`).

See the *Saving figures* section below for usage details.


In [ ]:
import os

# ======================================================================
# A4-aware Plotly figure export
#
# Targets an A4 (210 x 297 mm) Illustrator artboard as the max canvas.
# Illustrator imports SVG at 72 dpi, so sizing in mm at 72 dpi means a
# placement at 100% inside an A4 artboard renders at the true physical
# size you specified.
# ======================================================================
MM_PER_INCH = 25.4
SVG_DPI = 72            # Illustrator SVG import dpi
PNG_DPI = 300           # sidecar raster preview

def mm_to_px(mm, dpi=SVG_DPI):
    return int(round(mm * dpi / MM_PER_INCH))

# Physical-size presets (max footprint = A4 portrait, 210 x 297 mm)
A4_PORTRAIT_MM  = (210, 297)
A4_LANDSCAPE_MM = (297, 210)
PANEL_FULL_MM   = (180, 130)   # full-width manuscript panel
PANEL_HALF_MM   = (86, 70)     # half-width / single-column panel

FIG_OUT_DIR = "figures"


def save_fig(fig, name, *, mm=None, width_px=None, height_px=None,
             out_dir=None, formats=("svg", "png")):
    """Save a Plotly figure as SVG (vector, Illustrator-editable) + high-dpi PNG.

    Sizing:
        mm=(w, h)            -> sized to those mm at 72 dpi (preferred).
        width_px / height_px -> raw pixel size.
        neither              -> uses fig.layout.width / height.

    PNG is rendered at PNG_DPI / SVG_DPI (~4.17x) for crisp previews.
    Requires kaleido:  pip install -U kaleido
    If the static image backend (kaleido) is missing, writes an interactive HTML instead.
    """
    out_dir = out_dir if out_dir is not None else FIG_OUT_DIR
    os.makedirs(out_dir, exist_ok=True)
    if mm is not None:
        width_px = mm_to_px(mm[0])
        height_px = mm_to_px(mm[1])
    png_scale = PNG_DPI / SVG_DPI

    paths = []
    try:
        for fmt in formats:
            path = os.path.join(out_dir, f"{name}.{fmt}")
            kw = {}
            if width_px:  kw["width"] = width_px
            if height_px: kw["height"] = height_px
            if fmt == "png":
                kw["scale"] = png_scale
            fig.write_image(path, format=fmt, **kw)
            paths.append(path)
            print(f"  -> {path}")
    except Exception as e:
        path = os.path.join(out_dir, f"{name}.html")
        fig.write_html(path, include_plotlyjs="cdn")
        print(f"  [no static export: {type(e).__name__}] -> {path}")
        return [path]
    return paths


## 3. Plot functions

All seven UMAP / bar / heatmap builders. Each takes `df` and returns a
`plotly.graph_objects.Figure`. Run this cell once.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# UMAP — categorical colourings.  All of them draw through _grouped_scatter
# (helpers cell): draw="intermix" (default, SCATTER_DRAW) or "stacked".
# ═══════════════════════════════════════════════════════════════════════
def umap_by_sample(df, marker_size=6, opacity=0.8,
                   width=1000, height=700, draw=None, seed=None):
    """UMAP colored by sample. 13 colors grouped by tissue context."""
    df = _ensure_sample_short(df)
    d = df.dropna(subset=["sample_short"])
    order = _ordered_present(d["sample_short"].unique(),
                             [SHORT[s] for s in STACK_ORDER])
    fig = go.Figure()
    _grouped_scatter(fig, d, "umap_1", "umap_2", "sample_short", order,
                     SAMPLE_PALETTE, marker_size=marker_size, opacity=opacity,
                     draw=draw, seed=seed)
    fig.update_layout(**_base_layout(width, height,
                                      legend_title="<b>Sample</b>"))
    _apply_umap_axes(fig)
    return fig


# ═══════════════════════════════════════════════════════════════════════
# UMAP — by Leiden cluster (with centroid labels)
# ═══════════════════════════════════════════════════════════════════════
def umap_by_cluster(df, resolution=1.0,
                    marker_size=6, opacity=0.85,
                    width=900, height=700,
                    label_centroids=False,
                    centroid_method="median",
                    legend_marker_size=12,
                    draw=None, seed=None):
    """
    UMAP colored by Leiden cluster at the given resolution. Cluster ids are
    shown in the legend (bigger than marker size so they're readable).

    Set label_centroids=True for Scanpy-style on-plot labels if preferred.
    centroid_method: 'median' (robust) or 'mean' (classic).
    """
    leiden_col = f"leiden_r{resolution}"
    if leiden_col not in df.columns:
        raise KeyError(f"Column {leiden_col!r} not in dataframe")

    d = df.dropna(subset=[leiden_col]).copy()
    d["_cluster"] = d[leiden_col].astype(int)
    clusters = sorted(d["_cluster"].unique())
    cmap = build_cluster_cmap(clusters)
    counts = d["_cluster"].value_counts()

    fig = go.Figure()
    _grouped_scatter(fig, d, "umap_1", "umap_2", "_cluster", clusters, cmap,
                     names={c: str(int(c)) for c in clusters},
                     hover={c: f"<b>Cluster {int(c)}</b><br>n = {int(counts[c]):,}"
                            for c in clusters},
                     marker_size=marker_size, opacity=opacity,
                     draw=draw, seed=seed, biggest_first=False)

    if label_centroids:
        for c in clusters:
            sub = d[d["_cluster"] == c]
            if centroid_method == "median":
                cx, cy = sub["umap_1"].median(), sub["umap_2"].median()
            else:
                cx, cy = sub["umap_1"].mean(), sub["umap_2"].mean()
            fig.add_annotation(
                x=cx, y=cy, text=f"<b>{int(c)}</b>",
                showarrow=False,
                font=dict(size=12, color="#222", family=FONT_FAMILY),
                bgcolor="rgba(255,255,255,0.78)",
                bordercolor="rgba(0,0,0,0.22)",
                borderwidth=0.6, borderpad=2.5,
            )

    fig.update_layout(**_base_layout(width, height,
                                      legend_title="<b>Cluster</b>"))
    _apply_umap_axes(fig)
    return fig


# ═══════════════════════════════════════════════════════════════════════
# UMAP — by pattern (back / front / dorsal / unpatterned / ridged / whorl)
# ═══════════════════════════════════════════════════════════════════════
def umap_by_pattern(df, marker_size=6, opacity=0.75,
                    width=900, height=680, draw=None, seed=None):
    """
    UMAP colored by epithelial pattern category. Best for the 'do patterns
    separate in morphospace?' question — six colors, maximally readable.
    """
    d = df.dropna(subset=["pattern"])
    fig = go.Figure()
    _grouped_scatter(fig, d, "umap_1", "umap_2", "pattern",
                     _ordered_present(d["pattern"].unique(), PATTERN_ORDER),
                     PATTERN_PALETTE, marker_size=marker_size, opacity=opacity,
                     draw=draw, seed=seed)
    fig.update_layout(**_base_layout(width, height,
                                      legend_title="<b>Pattern</b>"))
    _apply_umap_axes(fig)
    return fig


# ═══════════════════════════════════════════════════════════════════════
# UMAP — by donor (embryo)
# ═══════════════════════════════════════════════════════════════════════
def umap_by_donor(df, marker_size=6, opacity=0.75,
                  width=900, height=680, draw=None, seed=None):
    """UMAP colored by donor (embryo). Useful for checking batch effects."""
    d = df.dropna(subset=["donor"])
    fig = go.Figure()
    _grouped_scatter(fig, d, "umap_1", "umap_2", "donor",
                     sorted(d["donor"].unique(), key=str),
                     DONOR_PALETTE, marker_size=marker_size, opacity=opacity,
                     draw=draw, seed=seed)
    fig.update_layout(**_base_layout(width, height,
                                      legend_title="<b>Donor</b>"))
    _apply_umap_axes(fig)
    return fig


# ═══════════════════════════════════════════════════════════════════════
# UMAP — by tissue (back / front / dorsal / volar)
# ═══════════════════════════════════════════════════════════════════════
def umap_by_tissue(df, marker_size=6, opacity=0.75,
                   width=860, height=680, draw=None, seed=None):
    """UMAP colored by tissue region (back / front / dorsal / volar)."""
    d = df.dropna(subset=["tissue"])
    fig = go.Figure()
    _grouped_scatter(fig, d, "umap_1", "umap_2", "tissue",
                     _ordered_present(d["tissue"].unique(), TISSUE_ORDER),
                     TISSUE_PALETTE, marker_size=marker_size, opacity=opacity,
                     draw=draw, seed=seed)
    fig.update_layout(**_base_layout(width, height,
                                      legend_title="<b>Tissue</b>"))
    _apply_umap_axes(fig)
    return fig


# ═══════════════════════════════════════════════════════════════════════
# UMAP — by timepoint (developmental stage)
# ═══════════════════════════════════════════════════════════════════════
def umap_by_timepoint(df, marker_size=6, opacity=0.75,
                      width=900, height=680, draw=None, seed=None):
    """
    UMAP colored by developmental timepoint. Uses a sequential (viridis-like)
    palette so time reads as an ordered gradient — earliest = yellow,
    latest = deep indigo.
    """
    d = df.dropna(subset=["timepoint"])
    fig = go.Figure()
    _grouped_scatter(fig, d, "umap_1", "umap_2", "timepoint",
                     _ordered_present(d["timepoint"].unique(), TIMEPOINT_ORDER),
                     TIMEPOINT_PALETTE, marker_size=marker_size, opacity=opacity,
                     draw=draw, seed=seed)
    fig.update_layout(**_base_layout(width, height,
                                      legend_title="<b>Timepoint</b>"))
    _apply_umap_axes(fig)
    return fig


# ═══════════════════════════════════════════════════════════════════════
# STACKED BAR — sample proportion per cluster
# ═══════════════════════════════════════════════════════════════════════
def cluster_proportion_bar(df, resolution=1.0,
                           width=1250, height=540,
                           bargap=0.18):
    """
    Stacked bar: for each Leiden cluster, the proportion contributed by
    each sample (column-normalized).
    """
    df = _ensure_sample_short(df)
    leiden_col = f"leiden_r{resolution}"
    sample_order = [SHORT[s] for s in STACK_ORDER
                    if SHORT[s] in df["sample_short"].unique()]

    prop = (df.groupby(["sample_short", leiden_col])
              .size().unstack(fill_value=0)
              .apply(lambda x: x / x.sum(), axis=0))
    prop = prop.reindex(sample_order)
    clusters_sorted = sorted(prop.columns.astype(int))

    fig = go.Figure()
    for sname in sample_order:
        if sname not in prop.index:
            continue
        vals = [prop.loc[sname, c] for c in clusters_sorted]
        fig.add_trace(go.Bar(
            x=[str(c) for c in clusters_sorted], y=vals,
            name=sname,
            marker=dict(color=SAMPLE_PALETTE.get(sname, "#888"),
                        line=dict(width=0.4, color="rgba(255,255,255,0.7)")),
            hovertemplate=(
                f"<b>{sname}</b><br>Cluster %{{x}}<br>"
                "Proportion: %{y:.3f}<extra></extra>"
            ),
        ))

    fig.update_layout(
        barmode="stack", bargap=bargap,
        **_base_layout(width, height, legend_title="<b>Sample</b>"),
    )
    fig.update_xaxes(
        type="category",
        title_text=f"Leiden cluster (resolution = {resolution})",
        title_font=dict(size=FONT_AXIS_TITLE, family=FONT_FAMILY),
        tickfont=dict(size=FONT_TICK, family=FONT_FAMILY),
        showgrid=False,
    )
    fig.update_yaxes(
        title_text="Proportion of cluster",
        title_font=dict(size=FONT_AXIS_TITLE, family=FONT_FAMILY),
        tickfont=dict(size=FONT_TICK, family=FONT_FAMILY),
        range=[0, 1.001], dtick=0.2,
        gridcolor="rgba(200,200,200,0.3)", gridwidth=0.5,
        zeroline=False,
    )
    return fig


# ═══════════════════════════════════════════════════════════════════════
# HEATMAP — sample × cluster proportions (alternative to the stacked bar)
# ═══════════════════════════════════════════════════════════════════════
def sample_cluster_heatmap(df, resolution=1.0,
                           normalize="sample",  # 'sample' | 'cluster'
                           width=1050, height=560):
    """
    Heatmap of proportions — optionally normalized per sample (rows sum to 1)
    or per cluster (columns sum to 1). Paired blues colorscale matches
    figure2_main.py.

    Often clearer than a stacked bar when the story is 'which clusters is
    each sample enriched in?'.
    """
    df = _ensure_sample_short(df)
    leiden_col = f"leiden_r{resolution}"
    sample_order = [SHORT[s] for s in STACK_ORDER
                    if SHORT[s] in df["sample_short"].unique()]

    counts = (df.groupby(["sample_short", leiden_col])
                .size().unstack(fill_value=0))
    counts = counts.reindex(sample_order)

    if normalize == "sample":
        Z = counts.div(counts.sum(axis=1), axis=0)
        cbar_title = "Fraction of sample"
    elif normalize == "cluster":
        Z = counts.div(counts.sum(axis=0), axis=1)
        cbar_title = "Fraction of cluster"
    else:
        raise ValueError("normalize must be 'sample' or 'cluster'")

    clusters_sorted = sorted(Z.columns.astype(int))
    Z = Z[clusters_sorted]

    colorscale = [
        [0.0, "#eff3ff"], [0.25, "#bdd7e7"],
        [0.5, "#6baed6"], [0.75, "#3182bd"], [1.0, "#08519c"],
    ]

    fig = go.Figure(go.Heatmap(
        z=Z.values,
        x=[str(c) for c in clusters_sorted],
        y=Z.index.tolist(),
        colorscale=colorscale,
        zmin=0, zmax=float(Z.values.max()),
        colorbar=dict(
            title=dict(text=cbar_title,
                       font=dict(size=FONT_AXIS_TITLE - 1, family=FONT_FAMILY)),
            thickness=16, len=0.75,
            tickfont=dict(size=FONT_TICK - 1, family=FONT_FAMILY),
            outlinewidth=0,
        ),
        hovertemplate=(
            "Sample: %{y}<br>Cluster: %{x}<br>"
            f"{cbar_title}: %{{z:.3f}}<extra></extra>"
        ),
    ))
    fig.update_xaxes(
        type="category",
        title_text=f"Leiden cluster (resolution = {resolution})",
        title_font=dict(size=FONT_AXIS_TITLE, family=FONT_FAMILY),
        tickfont=dict(size=FONT_TICK, family=FONT_FAMILY),
    )
    fig.update_yaxes(
        autorange="reversed",
        tickfont=dict(size=FONT_TICK, family=FONT_FAMILY),
    )
    fig.update_layout(
        plot_bgcolor="#fff", paper_bgcolor="#fff",
        width=width, height=height,
        margin=dict(l=300, r=80, t=30, b=70),
        font=dict(family=FONT_FAMILY),
    )
    return fig

---

# Visualizations

Below, one cell per figure. After running the setup cells above, each plot
cell is independent — run them in any order.

## UMAP — colored by donor (embryo)

Batch-effect check. If samples from the same embryo cluster together more than
samples from the same tissue type, that's a batch-effect signal worth flagging.

In [ ]:
fig = umap_by_donor(df)
fig.show()
# save_fig(fig, "umap_by_donor")  # native resolution; scale in Illustrator

## UMAP — colored by tissue

Four regions: back / front / dorsal (digit) / volar. The coarsest version of
the pattern story — four colors.

In [ ]:
fig = umap_by_tissue(df)
fig.show()
# save_fig(fig, "umap_by_tissue")  # native resolution; scale in Illustrator

## UMAP — colored by timepoint

Developmental stage from 12wk to 14wk with a sequential (viridis-like)
palette: yellow = earliest, deep indigo = latest.

In [ ]:
fig = umap_by_timepoint(df)
fig.show()
# save_fig(fig, "umap_by_timepoint")  # native resolution; scale in Illustrator

## Cluster proportion bar (column-normalized)

For each Leiden cluster, the proportion contributed by each sample. Each
column sums to 1. Good for spotting sample-specific clusters.

In [ ]:
fig = cluster_proportion_bar(df, resolution=RESOLUTION)
fig.show()
save_fig(fig, "cluster_proportion_bar")  # native resolution; scale in Illustrator

## Sample × cluster heatmap — sample-normalized

Each **row** sums to 1. Shows: *"of all cells in this sample, what fraction
landed in each cluster?"* This is each sample's **morphological fingerprint**.

Use this view to find sample signatures.

In [ ]:
fig = sample_cluster_heatmap(df, resolution=RESOLUTION, normalize='sample')
fig.show()
# save_fig(fig, "sample_cluster_heatmap_sample_norm")  # native resolution; scale in Illustrator

## Sample × cluster heatmap — cluster-normalized

Each **column** sums to 1. Same information as the stacked bar above, just in
heatmap form. *Redundant with the stacked bar — include only one in the final
figure.*

In [ ]:
fig = sample_cluster_heatmap(df, resolution=RESOLUTION, normalize='cluster')
fig.show()
# save_fig(fig, "sample_cluster_heatmap_cluster_norm")  # native resolution; scale in Illustrator

---

## Customization

All plot functions accept `marker_size`, `opacity`, `width`, `height` overrides:

```python
# Bigger markers, more opaque
umap_by_pattern(df, marker_size=8, opacity=0.9).show()

# Taller plot for a figure panel
umap_by_cluster(df, resolution=1.0, width=800, height=900).show()

# On-data cluster labels (Scanpy style)
umap_by_cluster(df, resolution=1.0, label_centroids=True).show()
```

---

## Saving figures for the paper

`save_fig(fig, name)` writes both an Illustrator-editable **SVG** (vector) and a
**high-DPI PNG** sidecar (300 dpi). **Default = native resolution** (whatever
width/height the plot function was tuned for, typically 900x680 px). SVG is
vector, so you scale it freely in Illustrator without quality loss.

If you need exact physical sizing for an A4 layout, pass the optional `mm=`
argument:

Presets:

| Preset             | Size (mm) | Use case                                 |
| ------------------ | --------- | ---------------------------------------- |
| `A4_PORTRAIT_MM`   | 210 x 297 | Full A4 page, portrait                   |
| `A4_LANDSCAPE_MM`  | 297 x 210 | Full A4 page, landscape                  |
| `PANEL_FULL_MM`    | 180 x 130 | Full-width manuscript panel              |
| `PANEL_HALF_MM`    | 86 x 70   | Half-width / single-column panel         |

Usage:

```python
fig = umap_by_pattern(df)
fig.show()
save_fig(fig, "umap_by_pattern")
# Native resolution (matches the figure function defaults).
# Scale freely in Illustrator -- SVG is vector, no quality loss.
# -> figures/umap_by_pattern.svg + figures/umap_by_pattern.png
```

Custom pixel size:

```python
save_fig(fig, "custom", width_px=900, height_px=600)
```

**Setup**: requires Plotly's static-image backend.

```bash
pip install -U kaleido
```


# Figure Panels — Main + Supplementary

Blocks below add:
- **A** UMAP with cluster median markers
- **B** Illustrator-ready UMAP (ghost background + highlighted medians)
- **C** Mesh gallery (dotted gridlines, vertical/horizontal orientation)

All panels reuse the palettes and typography from cells above.

## Load morphospace results

Pulls median representative info and mesh paths from `morphospace_results.pkl`.

In [ ]:
import os, pickle, gc

MORPHO_PKL = os.path.join('run', 'morphospace_results.pkl')

with open(MORPHO_PKL, 'rb') as f:
    morpho = pickle.load(f)

pcs = morpho['pcs']
morpho_config = morpho.get('config', {})
morpho_leiden = morpho.get('leiden_results', {})
CLUSTER_CMAP = morpho.get('CLUSTER_CMAP', {})

# Rebuild cluster cmap if needed
if not CLUSTER_CMAP:
    leiden_col = f"leiden_r{RESOLUTION}"
    if leiden_col in df.columns:
        CLUSTER_CMAP = build_cluster_cmap(set(df[leiden_col].dropna().astype(int)))

print(f"Morphospace pkl: {len(df):,} cells, {pcs.shape[1]} PCs")
print(f"Leiden resolutions: {list(morpho_leiden.keys())}")

## Helper: median representative selection

Finds the single cell closest to the cluster's component-wise median in
10D PC space. Returns a DataFrame with one row per cluster: label, stack,
UMAP coords, PC coords, and mesh path (if available).

In [ ]:
def get_median_representatives(df, pcs, resolution=1.0, mesh_lookup=None):
    """
    For each cluster, find the single cell closest to the cluster median
    in PC space (Euclidean distance to component-wise median).
    """
    leiden_col = f"leiden_r{resolution}"
    clusters = sorted(df[leiden_col].dropna().astype(int).unique())
    rows = []
    for c in clusters:
        mask = df[leiden_col].astype(int) == c
        idx = df.index[mask]
        if len(idx) == 0:
            continue
        pc_sub = pcs[mask.values]
        median_pt = np.median(pc_sub, axis=0)
        dists = np.sqrt(((pc_sub - median_pt) ** 2).sum(axis=1))
        best_local = int(np.argmin(dists))
        best_global = idx[best_local]
        row = df.loc[best_global]
        entry = {
            'cluster': int(c),
            'label': int(row['label']),
            'stack_name': row['stack_name'],
            'umap_1': row['umap_1'],
            'umap_2': row['umap_2'],
            'pc_dist': float(dists[best_local]),
        }
        if mesh_lookup:
            key = (row['stack_name'], int(row['label']))
            entry['mesh_path'] = mesh_lookup.get(key, '')
        rows.append(entry)
    return pd.DataFrame(rows)


# Build mesh lookup from manifest CSVs (same as notebook 01)
mesh_lookup = {}
import glob as _glob
for csv_path in _glob.glob('run/leiden_res_*/representative_cells.csv'):
    try:
        _rdf = pd.read_csv(csv_path)
        for _, _row in _rdf.iterrows():
            key = (str(_row.get('stack_name', '')), int(_row.get('label', 0)))
            mp = str(_row.get('mesh_path', ''))
            if mp and os.path.exists(mp):
                mesh_lookup[key] = mp
    except Exception:
        pass

# Also try the pkl mesh_lookup
if 'mesh_lookup' in morpho:
    for k, v in morpho['mesh_lookup'].items():
        if os.path.exists(str(v)):
            mesh_lookup[k] = str(v)

medians_df = get_median_representatives(df, pcs, resolution=RESOLUTION,
                                         mesh_lookup=mesh_lookup)
print(f"Median representatives: {len(medians_df)} clusters")
print(medians_df[['cluster', 'label', 'stack_name',
                   'umap_1', 'umap_2', 'pc_dist']].to_string(index=False))

## Block A — UMAP with cluster median markers

Standard cluster UMAP with median representative positions overlaid as
larger outlined diamond markers. Cluster ID labels at the median position.

In [ ]:
def umap_with_medians(df, medians_df, resolution=1.0,
                     marker_size=5, median_size=14, opacity=0.7,
                     width=950, height=750,
                     show_labels=True, draw=None, seed=None):
    """UMAP by cluster with median representative markers overlaid."""
    leiden_col = f"leiden_r{resolution}"
    clusters = sorted(df[leiden_col].dropna().astype(int).unique())
    cmap = build_cluster_cmap(clusters)

    fig = go.Figure()

    d = df.dropna(subset=[leiden_col]).copy()
    d["_cluster"] = d[leiden_col].astype(int)
    counts = d["_cluster"].value_counts()
    _grouped_scatter(fig, d, "umap_1", "umap_2", "_cluster", clusters, cmap,
                     names={c: f"C{int(c)} (n={int(counts[c]):,})" for c in clusters},
                     hover={c: f"<b>Cluster {int(c)}</b>" for c in clusters},
                     legendgroups={c: f"c{c}" for c in clusters},
                     marker_size=marker_size, opacity=opacity,
                     draw=draw, seed=seed, biggest_first=False)

    for _, mrow in medians_df.iterrows():
        c = int(mrow['cluster'])
        fig.add_trace(go.Scatter(
            x=[mrow['umap_1']], y=[mrow['umap_2']],
            mode="markers+text" if show_labels else "markers",
            marker=dict(
                size=median_size,
                color=cmap.get(c, '#888'),
                line=dict(width=2.5, color='#222'),
                symbol='diamond',
            ),
            text=[f"  {c}"] if show_labels else None,
            textposition="middle right",
            textfont=dict(size=11, color='#222', family=FONT_FAMILY),
            name=f"median C{c}",
            legendgroup=f"c{c}",
            showlegend=False,
            hovertemplate=(
                f"<b>Median C{c}</b><br>"
                f"label={int(mrow['label'])}<br>"
                f"stack={mrow['stack_name'][:30]}<br>"
                "UMAP1: %{x:.2f}<br>UMAP2: %{y:.2f}<extra></extra>"
            ),
        ))

    fig.update_layout(**_base_layout(width, height,
                                      legend_title="<b>Cluster</b>"))
    _apply_umap_axes(fig)
    return fig


fig = umap_with_medians(df, medians_df, resolution=RESOLUTION)
fig.show()
# save_fig(fig, "umap_with_medians")  # native resolution; scale in Illustrator

## Block B — Illustrator-ready UMAP (medians highlighted)

Background cells at low opacity, median representatives at full size +
black outline. Export as SVG, then place mesh insets with leader lines
in Illustrator.

Also exports `median_representatives.csv` with UMAP coordinates and mesh
paths for precise Illustrator placement.

In [ ]:
def umap_illustrator_ready(df, medians_df, resolution=1.0,
                         bg_size=3, bg_opacity=0.12,
                         median_size=16,
                         width=950, height=750):
    """
    Ghost UMAP: background at very low opacity, median representatives
    at full opacity with cluster color + thick black outline.
    Designed for Illustrator overlay with mesh insets.
    """
    leiden_col = f"leiden_r{resolution}"
    clusters = sorted(df[leiden_col].dropna().astype(int).unique())
    cmap = build_cluster_cmap(clusters)

    fig = go.Figure()

    # Ghost background
    fig.add_trace(go.Scatter(
        x=df["umap_1"], y=df["umap_2"], mode="markers",
        marker=dict(size=bg_size, color='#BBBBBB',
                    opacity=bg_opacity, line=dict(width=0)),
        name="all cells",
        showlegend=False,
        hoverinfo='skip',
    ))

    # Median markers
    for _, mrow in medians_df.iterrows():
        c = int(mrow['cluster'])
        fig.add_trace(go.Scatter(
            x=[mrow['umap_1']], y=[mrow['umap_2']],
            mode="markers+text",
            marker=dict(
                size=median_size,
                color=cmap.get(c, '#888'),
                line=dict(width=3, color='#111'),
                symbol='diamond',
            ),
            text=[f"  C{c}"],
            textposition="middle right",
            textfont=dict(size=12, color='#333', family=FONT_FAMILY),
            name=f"C{c}",
            hovertemplate=(
                f"<b>Median C{c}</b><br>"
                f"label={int(mrow['label'])}<br>"
                f"{mrow['stack_name'][:35]}<extra></extra>"
            ),
        ))

    fig.update_layout(**_base_layout(width, height,
                                      legend_title="<b>Representative</b>"))
    _apply_umap_axes(fig)
    return fig


fig_ai = umap_illustrator_ready(df, medians_df, resolution=RESOLUTION)
fig_ai.show()
# save_fig(fig_ai, "umap_illustrator_ready")  # native resolution; scale in Illustrator

# Export median info for Illustrator
out_csv = os.path.join('run', 'median_representatives.csv')
medians_df.to_csv(out_csv, index=False)
print(f"Saved: {out_csv}")
print(f"Columns: {list(medians_df.columns)}")

## Block C — Mesh gallery

Renders representative cell meshes with PyVista. Two modes:
- `orientation="vertical"` — rows = clusters, columns = representatives
- `orientation="horizontal"` — columns = clusters, rows = representatives

Subtle dotted gray gridlines between cells. White background for print.

**Main figure**: 1 cell per cluster (the median) — single row.
**Supplementary**: 10 cells per cluster — full grid.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

try:
    import pyvista as pv
    import trimesh
    pv.OFF_SCREEN = True
    HAS_PYVISTA = True
except ImportError:
    HAS_PYVISTA = False
    print("PyVista/trimesh not available -- gallery rendering disabled")


def render_cell(mesh_path, color='#4ECDC4',
                camera_position='iso',
                window_size=(800, 800),
                background='#FFFFFF',
                smooth_iter=30):
    """Render a cell .obj mesh with PyVista 3-point lighting. Returns RGBA array."""
    tm = trimesh.load(mesh_path, file_type='obj', process=False)
    verts = np.array(tm.vertices, dtype=float)
    faces = np.array(tm.faces, dtype=int)
    verts = verts[:, [2, 1, 0]]  # ZYX -> XYZ
    verts -= verts.mean(axis=0)
    rms = np.sqrt((verts**2).sum(axis=1).mean())
    verts /= (rms * 1.8)

    pv_faces = np.hstack([np.full((len(faces), 1), 3), faces]).ravel()
    mesh = pv.PolyData(verts, pv_faces)
    if smooth_iter > 0:
        mesh = mesh.smooth(n_iter=smooth_iter, relaxation_factor=0.08)

    cam_cfgs = {
        'iso': dict(pos=(2.8, 2.2, 1.6), up=(0, 0, 1)),
        'xz':  dict(pos=(0.0, -3.2, 0.0), up=(0, 0, 1)),
        'xy':  dict(pos=(0.0, 0.0, 3.2), up=(0, 1, 0)),
        'yz':  dict(pos=(3.2, 0.0, 0.0), up=(0, 0, 1)),
    }
    cfg = cam_cfgs.get(camera_position, cam_cfgs['iso'])

    pl = pv.Plotter(off_screen=True, window_size=window_size)
    pl.set_background(background)
    pl.add_mesh(mesh, color=color, smooth_shading=True,
                specular=0.45, specular_power=25,
                ambient=0.15, diffuse=0.85)
    pl.camera.position = cfg['pos']
    pl.camera.focal_point = (0, 0, 0)
    pl.camera.up = cfg['up']
    pl.remove_all_lights()
    pl.add_light(pv.Light(position=(3.5, 3.0, 5.0), intensity=0.80))
    pl.add_light(pv.Light(position=(-2.0, -2.0, 2.0), intensity=0.28))
    pl.add_light(pv.Light(position=(0.0, 2.0, -2.5), intensity=0.18))
    img = pl.screenshot(None, return_img=True)
    pl.close()
    return img


def build_gallery(thumbnails, cluster_ids, cmap,
                  n_cols_per_cluster=None,
                  orientation='vertical',
                  cell_size=2.5,
                  save_path=None,
                  dpi=600):
    """
    Build a mesh gallery figure.

    thumbnails: dict of {(cluster_id, rep_index): RGBA array}
    cluster_ids: sorted list of cluster IDs
    cmap: {cluster_id: hex_color}
    n_cols_per_cluster: reps shown per cluster (auto-detected if None)
    orientation: 'vertical' (rows=clusters) or 'horizontal' (cols=clusters)
    """
    if n_cols_per_cluster is None:
        max_rep = max(idx for (_, idx) in thumbnails.keys()) + 1 if thumbnails else 1
        n_cols_per_cluster = max_rep

    n_clusters = len(cluster_ids)
    if orientation == 'vertical':
        n_rows, n_cols = n_clusters, n_cols_per_cluster
    else:
        n_rows, n_cols = n_cols_per_cluster, n_clusters

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(cell_size * n_cols, cell_size * n_rows),
        facecolor='white',
    )
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    if n_cols == 1:
        axes = axes.reshape(-1, 1)

    for ax in axes.flat:
        ax.set_facecolor('white')
        ax.set_xticks([]); ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_linestyle(':')
            sp.set_linewidth(0.7)
            sp.set_color('#BBBBBB')

    for ci, c in enumerate(cluster_ids):
        color_hex = cmap.get(c, '#888888')
        for ri in range(n_cols_per_cluster):
            if orientation == 'vertical':
                row_idx, col_idx = ci, ri
            else:
                row_idx, col_idx = ri, ci

            ax = axes[row_idx, col_idx]
            key = (c, ri)
            if key in thumbnails:
                ax.imshow(thumbnails[key])
            else:
                ax.text(0.5, 0.5, '--', ha='center', va='center',
                        fontsize=9, color='#999', transform=ax.transAxes)

            if orientation == 'vertical' and ri == 0:
                ax.set_ylabel(f'C{c}', fontsize=10, color=color_hex,
                              fontweight='bold', labelpad=4,
                              fontfamily=FONT_FAMILY)
            elif orientation == 'horizontal' and ri == 0:
                ax.set_title(f'C{c}', fontsize=10, color=color_hex,
                             fontweight='bold', pad=4,
                             fontfamily=FONT_FAMILY)

    plt.tight_layout(pad=0.3)
    if save_path:
        # TIFF gets LZW compression (Illustrator + Photoshop read it transparently)
        sp = str(save_path)
        kw = dict(dpi=dpi, bbox_inches='tight', facecolor='white')
        if sp.lower().endswith(('.tif', '.tiff')):
            kw['pil_kwargs'] = {'compression': 'tiff_lzw'}
        plt.savefig(sp, **kw)
        print(f"Saved: {sp}")
    plt.show()
    return fig


if HAS_PYVISTA:
    print("render_cell() + build_gallery() defined")
else:
    print("Gallery functions defined (PyVista unavailable)")

### Main figure gallery — 1 median per cluster

Single row of median representative meshes, one per cluster. For the
fanned-out inset around the UMAP in the main figure.

Individual PNGs are also saved for free placement in Illustrator.

In [ ]:
if HAS_PYVISTA:
    leiden_col = f"leiden_r{RESOLUTION}"
    clusters = sorted(df[leiden_col].dropna().astype(int).unique())
    cmap = build_cluster_cmap(clusters)

    median_thumbs = {}
    individual_dir = os.path.join('run', 'median_renders')
    os.makedirs(individual_dir, exist_ok=True)

    for _, mrow in medians_df.iterrows():
        c = int(mrow['cluster'])
        mp = mrow.get('mesh_path', '')
        if not mp or not os.path.exists(str(mp)):
            print(f"  C{c}: no mesh found, skipping")
            continue
        color = cmap.get(c, '#888888')
        try:
            img = render_cell(str(mp), color=color, camera_position='iso')
            median_thumbs[(c, 0)] = img
            ind_path = os.path.join(individual_dir, f'median_C{c}.tif')
            from PIL import Image
            Image.fromarray(img).save(ind_path, format='TIFF', compression='tiff_lzw')
        except Exception as e:
            print(f"  C{c}: render error: {e}")

    print(f"Rendered {len(median_thumbs)} median cells")

    if median_thumbs:
        build_gallery(
            median_thumbs, clusters, cmap,
            n_cols_per_cluster=1,
            orientation='horizontal',
            cell_size=3.0,
            save_path=os.path.join('run',
                                   'median_gallery_main.tif'),
            dpi=600,
        )
        print(f"Individual renders: {individual_dir}/")
else:
    print("PyVista not available -- skip gallery rendering")

### Supplementary gallery — 10 representatives per cluster

Full grid: rows = clusters, columns = 10 closest-to-median cells.
Dotted gray gridlines. Saved at high DPI for supplementary figure.

In [ ]:
if HAS_PYVISTA:
    rep_csv_path = os.path.join('run',
                                f'leiden_res_{RESOLUTION}',
                                'representative_cells.csv')
    if os.path.exists(rep_csv_path):
        rep_df = pd.read_csv(rep_csv_path)
        rep_df = rep_df[rep_df.get('mesh_exists', True) == True]

        leiden_col = f"leiden_r{RESOLUTION}"
        clusters = sorted(df[leiden_col].dropna().astype(int).unique())
        cmap = build_cluster_cmap(clusters)

        supp_thumbs = {}
        for _, row in rep_df.iterrows():
            c = int(row['cluster'])
            role = str(row.get('role', ''))
            rep_idx = int(role.replace('rep_', '')) if role.startswith('rep_') else 0
            mp = str(row.get('mesh_path', ''))
            if not mp or not os.path.exists(mp):
                continue
            color = cmap.get(c, '#888888')
            try:
                supp_thumbs[(c, rep_idx)] = render_cell(mp, color=color)
            except Exception as e:
                print(f"  C{c} rep_{rep_idx}: {e}")
            if len(supp_thumbs) % 20 == 0:
                print(f"  Rendered {len(supp_thumbs)} / {len(rep_df)} ...")

        print(f"Rendered {len(supp_thumbs)} thumbnails")

        if supp_thumbs:
            build_gallery(
                supp_thumbs, clusters, cmap,
                orientation='horizontal',
                cell_size=2.5,
                save_path=os.path.join('run',
                                       f'cluster_gallery_supp_res{RESOLUTION}.tif'),
                dpi=600,
            )
    else:
        print(f"No representative CSV found at {rep_csv_path}")
else:
    print("PyVista not available -- skip supplementary gallery")

gc.collect()